# Discovering Communities in a Real-World Network Using Louvain and Girvan-Newman

**Dataset:** `socfb-Caltech36`, a Facebook friendship network from NetworkRepository.

**Goal:** Prepare the graph, analyze its structural properties, apply Louvain and Girvan-Newman community detection, then compare the detected communities by modularity, size, runtime, and visual separation.


## 1. Setup

The notebook installs missing packages only when they are not already available. This keeps the project reproducible while avoiding unnecessary installs on a prepared environment.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "networkx": "networkx",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "community": "python-louvain",
    "requests": "requests",
}

missing = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")


All required packages are available.


In [2]:
import io
import statistics
import time
import zipfile
from pathlib import Path

import community as community_louvain
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from networkx.algorithms.community import modularity
from networkx.algorithms.community.centrality import girvan_newman

RANDOM_SEED = 42
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FIGURE_DIR = PROJECT_DIR / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

DATASET_NAME = "socfb-Caltech36"
DATASET_PAGE = "https://networkrepository.com/socfb_Caltech36.php"
DATASET_ZIP_URL = "https://nrvis.com/download/data/socfb/socfb-Caltech36.zip"
DATASET_ZIP_PATH = DATA_DIR / "socfb-Caltech36.zip"
MATRIX_MEMBER = "socfb-Caltech36.mtx"

plt.style.use("seaborn-v0_8-whitegrid")


## 2. Data Collection and Graph Preparation

NetworkRepository provides this graph in MatrixMarket format. The file is marked as symmetric, so each listed coordinate is interpreted as one undirected friendship edge.


In [3]:
def download_dataset(url: str, output_path: Path) -> None:
    if output_path.exists():
        print(f"Using cached dataset: {output_path}")
        return

    response = requests.get(url, timeout=60)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    print(f"Downloaded dataset to: {output_path}")


def load_symmetric_matrix_market_graph(zip_path: Path, member_name: str) -> nx.Graph:
    graph = nx.Graph()

    with zipfile.ZipFile(zip_path) as archive:
        raw_text = archive.read(member_name).decode("utf-8")

    dimensions_read = False
    for raw_line in raw_text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("%"):
            continue

        parts = line.split()
        if not dimensions_read:
            node_count = int(parts[0])
            graph.add_nodes_from(range(1, node_count + 1))
            dimensions_read = True
            continue

        if len(parts) >= 2:
            source, target = int(parts[0]), int(parts[1])
            if source != target:
                graph.add_edge(source, target)

    return graph


download_dataset(DATASET_ZIP_URL, DATASET_ZIP_PATH)
G = load_symmetric_matrix_market_graph(DATASET_ZIP_PATH, MATRIX_MEMBER)
components = sorted(nx.connected_components(G), key=len, reverse=True)
G_lcc = G.subgraph(components[0]).copy()

print(f"Dataset: {DATASET_NAME}")
print(f"Source page: {DATASET_PAGE}")
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Unique undirected edges: {G.number_of_edges():,}")
print(f"Connected components: {len(components):,}")
print(f"Largest connected component: {G_lcc.number_of_nodes():,} nodes, {G_lcc.number_of_edges():,} edges")


Using cached dataset: /Users/kris/Desktop/3283/project2/data/socfb-Caltech36.zip
Dataset: socfb-Caltech36
Source page: https://networkrepository.com/socfb_Caltech36.php
Nodes: 769
Unique undirected edges: 16,656
Connected components: 4
Largest connected component: 762 nodes, 16,651 edges


In [4]:
degrees = dict(G.degree())
lcc_degrees = dict(G_lcc.degree())

basic_properties = pd.DataFrame(
    [
        ["Nodes", G.number_of_nodes()],
        ["Unique undirected edges", G.number_of_edges()],
        ["Average degree", np.mean(list(degrees.values()))],
        ["Minimum degree", min(degrees.values())],
        ["Maximum degree", max(degrees.values())],
        ["Density", nx.density(G)],
        ["Connected components", nx.number_connected_components(G)],
        ["Largest component nodes", G_lcc.number_of_nodes()],
        ["Largest component edges", G_lcc.number_of_edges()],
        ["Largest component average shortest path length", nx.average_shortest_path_length(G_lcc)],
        ["Largest component diameter", nx.diameter(G_lcc)],
        ["Largest component radius", nx.radius(G_lcc)],
    ],
    columns=["Property", "Value"],
)

display(basic_properties)


,Property,Value
0,Nodes,769.000000
1,Unique undirected edges,16656.000000
2,Average degree,43.318596
3,Minimum degree,1.000000
4,Maximum degree,248.000000
5,Density,0.056404
6,Connected components,4.000000
7,Largest component nodes,762.000000
8,Largest component edges,16651.000000
9,Largest component average shortest path length,2.337848


### Reading the Basic Network Properties

The first important point is that this is not a very sparse online network. The dataset contains **769 nodes** and **16,656 unique undirected friendship edges**. Its density is **0.0564**, so about **5.64%** of all possible student pairs are connected. For a real social network, this is quite dense, which makes sense because Caltech is a relatively small university environment where many students can know each other through residence, classes, clubs, and shared friend groups.

The average degree is **43.32**, meaning that a student in this graph has about **43 Facebook friends** inside this Caltech network. However, the maximum degree is **248**, while the minimum degree is only **1**. This wide gap already suggests a strong difference between ordinary users and highly connected social hubs. The graph has **4 connected components**, but the largest connected component contains **762 out of 769 nodes**, or about **99.1%** of the whole network. The remaining components are very small: **3, 2, and 2 nodes**. Therefore, almost the entire campus friendship network belongs to one main social structure, with only a few tiny peripheral groups separated from it.

The largest component has an average shortest path length of **2.34**, diameter **6**, and radius **4**. This means the graph is not only dense, but also compact: most users can reach most other users through only a few friendship steps. This is why the later shortest-path and community visualizations focus on the largest connected component when a connected graph is required.


In [5]:
layout_start = time.perf_counter()
pos_lcc = nx.spring_layout(G_lcc, seed=RANDOM_SEED, iterations=60, k=0.12)
layout_time = time.perf_counter() - layout_start

node_sizes = [20 + 180 * (lcc_degrees[node] / max(lcc_degrees.values())) for node in G_lcc.nodes()]

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(G_lcc, pos_lcc, ax=ax, width=0.25, alpha=0.12, edge_color="#555555")
nodes = nx.draw_networkx_nodes(
    G_lcc,
    pos_lcc,
    ax=ax,
    node_color=[lcc_degrees[node] for node in G_lcc.nodes()],
    node_size=node_sizes,
    cmap="viridis",
    linewidths=0,
    alpha=0.9,
)
ax.set_title("socfb-Caltech36 largest connected component, colored by degree")
ax.axis("off")
cbar = fig.colorbar(nodes, ax=ax, shrink=0.75)
cbar.set_label("Degree")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_graph_degree_visualization.png", dpi=180)
plt.close(fig)

print(f"Spring layout runtime: {layout_time:.2f} seconds")


Spring layout runtime: 1.32 seconds


![Graph visualization colored by degree](figures/01_graph_degree_visualization.png)


## 3. Centrality Analysis

The centrality results below are useful because they identify which node IDs behave like social hubs or bridges in this particular network. Degree centrality shows direct popularity, betweenness centrality shows how often a node lies on shortest paths, closeness centrality measures how quickly a node can reach others, and eigenvector centrality gives more weight to nodes connected to other influential nodes.

In the interpretation, I focus especially on whether the same nodes appear across several measures. If one node is high in degree, closeness, and eigenvector centrality, it is probably part of the social core. If a node has high betweenness, it may be more important as a connector between groups, even if it is not the highest-degree user.


In [6]:
centrality_start = time.perf_counter()
degree_centrality = nx.degree_centrality(G_lcc)
betweenness_centrality = nx.betweenness_centrality(G_lcc)
closeness_centrality = nx.closeness_centrality(G_lcc)
eigenvector_centrality = nx.eigenvector_centrality(G_lcc, max_iter=1000)
centrality_runtime = time.perf_counter() - centrality_start

centrality_df = pd.DataFrame(
    {
        "node": list(G_lcc.nodes()),
        "degree": [degree_centrality[n] for n in G_lcc.nodes()],
        "betweenness": [betweenness_centrality[n] for n in G_lcc.nodes()],
        "closeness": [closeness_centrality[n] for n in G_lcc.nodes()],
        "eigenvector": [eigenvector_centrality[n] for n in G_lcc.nodes()],
        "raw_degree": [G_lcc.degree(n) for n in G_lcc.nodes()],
    }
)

print(f"Centrality runtime: {centrality_runtime:.2f} seconds")
print("Top nodes by each centrality measure:")
for metric in ["degree", "betweenness", "closeness", "eigenvector"]:
    print(f"\nTop 8 by {metric}:")
    display(centrality_df.sort_values(metric, ascending=False).head(8)[["node", "raw_degree", metric]])

top_degree_nodes = centrality_df.sort_values("degree", ascending=False).head(8)["node"].tolist()
top_degree_betweenness = centrality_df[centrality_df["node"].isin(top_degree_nodes)][
    ["node", "raw_degree", "degree", "betweenness"]
].sort_values("degree", ascending=False)

print("\nBetweenness values for the 8 highest-degree nodes:")
display(top_degree_betweenness)


Centrality runtime: 2.19 seconds
Top nodes by each centrality measure:

Top 8 by degree:


,node,raw_degree,degree
701,709,248,0.325887
86,90,203,0.266754
216,223,194,0.254928
656,664,184,0.241787
250,257,172,0.226018
271,278,171,0.224704
679,687,160,0.210250
615,623,160,0.210250



Top 8 by betweenness:


,node,raw_degree,betweenness
701,709,248,0.047949
216,223,194,0.043367
271,278,171,0.030954
86,90,203,0.029342
348,355,144,0.022289
727,735,152,0.021845
630,638,157,0.018686
656,664,184,0.018633



Top 8 by closeness:


,node,raw_degree,closeness
701,709,248,0.574773
86,90,203,0.553052
216,223,194,0.543960
271,278,171,0.538570
679,687,160,0.537050
656,664,184,0.535162
250,257,172,0.532540
727,735,152,0.531054



Top 8 by eigenvector:


,node,raw_degree,eigenvector
701,709,248,0.131991
86,90,203,0.116052
615,623,160,0.114047
250,257,172,0.111609
656,664,184,0.107220
216,223,194,0.100884
679,687,160,0.097354
370,377,144,0.095733



Betweenness values for the 8 highest-degree nodes:


,node,raw_degree,degree,betweenness
701,709,248,0.325887,0.047949
86,90,203,0.266754,0.029342
216,223,194,0.254928,0.043367
656,664,184,0.241787,0.018633
250,257,172,0.226018,0.012856
271,278,171,0.224704,0.030954
615,623,160,0.210250,0.009810
679,687,160,0.210250,0.014301


In [7]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metrics = ["degree", "betweenness", "closeness", "eigenvector"]
colors = ["#2a9d8f", "#e76f51", "#457b9d", "#7b2cbf"]

for ax, metric, color in zip(axes.flatten(), metrics, colors):
    ax.hist(centrality_df[metric], bins=30, color=color, alpha=0.85)
    ax.set_title(f"{metric.title()} centrality distribution")
    ax.set_xlabel(metric.title())
    ax.set_ylabel("Node count")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_centrality_distributions.png", dpi=180)
plt.close(fig)

top_betweenness_nodes = centrality_df.sort_values("betweenness", ascending=False).head(10)["node"].tolist()
highlight_nodes = set(top_degree_nodes) | set(top_betweenness_nodes)
node_colors = [
    "#e76f51" if node in top_betweenness_nodes else "#2a9d8f" if node in top_degree_nodes else "#b8c0cc"
    for node in G_lcc.nodes()
]
node_sizes = [180 if node in highlight_nodes else 18 for node in G_lcc.nodes()]

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(G_lcc, pos_lcc, ax=ax, width=0.25, alpha=0.10, edge_color="#555555")
nx.draw_networkx_nodes(G_lcc, pos_lcc, ax=ax, node_color=node_colors, node_size=node_sizes, linewidths=0.2, edgecolors="#222222")
nx.draw_networkx_labels(
    G_lcc.subgraph(highlight_nodes),
    pos_lcc,
    ax=ax,
    font_size=8,
    font_color="#111111",
)
ax.set_title("Highest degree and betweenness centrality nodes")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_high_centrality_nodes.png", dpi=180)
plt.close(fig)


![Centrality distributions](figures/02_centrality_distributions.png)

![High centrality nodes](figures/03_high_centrality_nodes.png)


### What the Centrality Tables Show

The strongest node in this network is clearly **node 709**. It has a raw degree of **248**, which means it is directly connected to almost one third of the largest connected component. Its degree centrality is **0.326**, closeness centrality is **0.575**, and eigenvector centrality is **0.132**, and it ranks first in all three measures. This suggests that node 709 is not just popular in a simple counting sense. It is also close to the rest of the graph and connected to other well-connected users, so it sits inside the main social core rather than on the edge.

The next major hubs are **node 90** with degree **203**, **node 223** with degree **194**, **node 664** with degree **184**, and **node 257** with degree **172**. These IDs repeatedly appear near the top of the degree, closeness, and eigenvector rankings, which supports the idea that the network has a small group of highly visible users. In Figure 3, these high-centrality nodes are located near the dense center of the spring layout instead of being scattered randomly around the outside.

The betweenness result adds a more interesting detail. Node 709 is still first in betweenness (**0.0479**), but node **223** is close behind (**0.0434**) even though it has fewer direct friends. Node **278** also has a relatively high betweenness score (**0.0310**) compared with some nodes that have similar or higher degree. This means that bridging power is not exactly the same as popularity. Some users may be important because they connect parts of the friendship network, not only because they have many direct friends.

The distribution plots in Figure 2 also show a strong long-tail pattern. Most nodes have low betweenness, while only a small number of users have visible bridge-like importance. At the same time, the highest betweenness scores are not extremely large. This is probably because the network is dense and highly clustered, so there are many alternative routes between students. In other words, even the biggest hub does not completely control information flow because the graph contains many redundant friendship paths.


## 4. Shortest Path and Distance Structure


In [8]:
all_lengths = dict(nx.all_pairs_shortest_path_length(G_lcc))
pair_distances = [
    distance
    for source, target_lengths in all_lengths.items()
    for target, distance in target_lengths.items()
    if source < target
]

representative_node = int(centrality_df.sort_values("degree", ascending=False).iloc[0]["node"])
distances_from_representative = list(all_lengths[representative_node].values())

shortest_path_summary = pd.DataFrame(
    [
        ["Average shortest path length", nx.average_shortest_path_length(G_lcc)],
        ["Median pairwise shortest path length", statistics.median(pair_distances)],
        ["Maximum shortest path length / diameter", max(pair_distances)],
        [f"Average distance from node {representative_node}", np.mean(distances_from_representative)],
    ],
    columns=["Metric", "Value"],
)
display(shortest_path_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(pair_distances, bins=np.arange(0.5, max(pair_distances) + 1.5, 1), color="#457b9d", alpha=0.85)
axes[0].set_title("All-pairs shortest path length distribution")
axes[0].set_xlabel("Shortest path length")
axes[0].set_ylabel("Number of node pairs")

axes[1].hist(
    distances_from_representative,
    bins=np.arange(-0.5, max(distances_from_representative) + 1.5, 1),
    color="#f4a261",
    alpha=0.90,
)
axes[1].set_title(f"Distances from high-degree node {representative_node}")
axes[1].set_xlabel("Shortest path length")
axes[1].set_ylabel("Number of nodes")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_shortest_path_distributions.png", dpi=180)
plt.close(fig)


,Metric,Value
0,Average shortest path length,2.337848
1,Median pairwise shortest path length,2.000000
2,Maximum shortest path length / diameter,6.000000
3,Average distance from node 709,1.737533


![Shortest path distributions](figures/04_shortest_path_distributions.png)


### Shortest Paths and the Small-World Pattern

The shortest-path results show a strong small-world pattern. In the largest connected component, the average shortest path length is only **2.3378**, and the median pairwise shortest path length is **2**. Out of all node pairs in the largest component, about **58.69%** are exactly **2 steps** apart and another **31.75%** are **3 steps** apart. Only a very small fraction need 5 or 6 steps. The diameter is **6**, so even the two most distant students in the main component are still connected within six friendship links.

Node **709** makes this compactness even more visible. Its average distance to the rest of the largest component is only **1.7375**. From node 709, there are **248 nodes** at distance 1 and **464 nodes** at distance 2, so more than **93%** of the largest component is within two steps of this one user. This supports the centrality result above: node 709 is not simply a high-degree node, but a node that can reach almost the whole social network very quickly.

In a campus setting, this has a concrete meaning. Information such as event invitations, class news, or rumors could spread very quickly because the network has both high-degree hubs and short path lengths. Figure 4 therefore does not just show a mathematical distribution; it shows that the social distance inside this Facebook network is very small.


## 5. Density, Degree Distribution, Clustering, Triangles, and Bridges


In [9]:
component_sizes = sorted([len(component) for component in nx.connected_components(G)], reverse=True)
degree_values = list(degrees.values())

component_degree_summary = pd.DataFrame(
    [
        ["Graph density", nx.density(G)],
        ["Number of connected components", nx.number_connected_components(G)],
        ["Component sizes", component_sizes],
        ["Mean degree", np.mean(degree_values)],
        ["Median degree", statistics.median(degree_values)],
        ["Maximum degree", max(degree_values)],
    ],
    columns=["Metric", "Value"],
)
display(component_degree_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(degree_values, bins=30, color="#2a9d8f", alpha=0.85)
axes[0].set_title("Degree distribution")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Node count")

axes[1].bar(range(1, len(component_sizes) + 1), component_sizes, color="#6d597a", alpha=0.90)
axes[1].set_title("Connected component sizes")
axes[1].set_xlabel("Component rank")
axes[1].set_ylabel("Nodes")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_degree_and_components.png", dpi=180)
plt.close(fig)


,Metric,Value
0,Graph density,0.056404
1,Number of connected components,4
2,Component sizes,"[762, 3, 2, 2]"
3,Mean degree,43.318596
4,Median degree,36
5,Maximum degree,248


![Degree and connected component distributions](figures/05_degree_and_components.png)


In [10]:
clustering_coefficients = nx.clustering(G)
triangle_counts = nx.triangles(G)
bridge_edges = list(nx.bridges(G))
local_bridge_edges = list(nx.local_bridges(G, with_span=False))

clustering_summary = pd.DataFrame(
    [
        ["Average clustering coefficient", nx.average_clustering(G)],
        ["Mean triangles per node", statistics.mean(triangle_counts.values())],
        ["Median triangles per node", statistics.median(triangle_counts.values())],
        ["Maximum triangles for one node", max(triangle_counts.values())],
        ["Number of bridges", len(bridge_edges)],
        ["Number of local bridges", len(local_bridge_edges)],
    ],
    columns=["Metric", "Value"],
)
display(clustering_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(list(clustering_coefficients.values()), bins=30, color="#457b9d", alpha=0.85)
axes[0].set_title("Clustering coefficient distribution")
axes[0].set_xlabel("Clustering coefficient")
axes[0].set_ylabel("Node count")

axes[1].hist(list(triangle_counts.values()), bins=30, color="#e76f51", alpha=0.85)
axes[1].set_title("Triangle count distribution")
axes[1].set_xlabel("Triangles")
axes[1].set_ylabel("Node count")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_clustering_and_triangles.png", dpi=180)
plt.close(fig)

lcc_bridge_edges = [edge for edge in bridge_edges if edge[0] in G_lcc and edge[1] in G_lcc]
lcc_local_bridge_edges = [edge for edge in local_bridge_edges if edge[0] in G_lcc and edge[1] in G_lcc]

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(G_lcc, pos_lcc, ax=ax, width=0.20, alpha=0.08, edge_color="#555555")
nx.draw_networkx_edges(G_lcc, pos_lcc, edgelist=lcc_local_bridge_edges, ax=ax, width=0.60, alpha=0.35, edge_color="#f4a261")
nx.draw_networkx_edges(G_lcc, pos_lcc, edgelist=lcc_bridge_edges, ax=ax, width=1.40, alpha=0.95, edge_color="#d62828")
nx.draw_networkx_nodes(G_lcc, pos_lcc, ax=ax, node_size=12, node_color="#b8c0cc", linewidths=0)
ax.set_title("Bridges and local bridges in the largest connected component")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_bridges_and_local_bridges.png", dpi=180)
plt.close(fig)


,Metric,Value
0,Average clustering coefficient,0.409294
1,Mean triangles per node,466.435631
2,Median triangles per node,259.000000
3,Maximum triangles for one node,3953.000000
4,Number of bridges,33.000000
5,Number of local bridges,100.000000


![Clustering and triangle distributions](figures/06_clustering_and_triangles.png)

![Bridges and local bridges](figures/07_bridges_and_local_bridges.png)


### Clustering, Triangles, and Bridge Edges

The clustering result is one of the clearest signs that this is a real social network rather than a random set of edges. The average clustering coefficient is **0.4093**, meaning that if a student is friends with two other students, there is a fairly high chance that those two friends are also friends with each other. This fits the real-life idea of residence groups, class groups, and club circles where mutual friends are common.

The triangle counts support the same point. On average, each node participates in **466.44 triangles**, with a median of **259**. The maximum triangle count is **3,953**, and this maximum belongs to **node 709**, the same node that dominated the centrality measures. This is important because it means node 709 is not only connected to many people; many of those people are also connected among themselves. That is exactly what we would expect from a user embedded in a dense campus friendship core.

The bridge results show the opposite side of the structure. The graph contains **33 bridges** and **100 local bridges**. A strict bridge is an edge whose removal would disconnect part of the graph, while a local bridge connects two nodes that do not share a common neighbor. In this dataset, **31 bridges** and **98 local bridges** are inside the largest connected component, so they are not only caused by the tiny external components. They also reveal fragile one-edge connections and cross-neighborhood ties inside the main network.

Figure 7 helps explain why these edges matter. Most of the graph is triangle-rich and redundant, but bridge and local-bridge edges represent weaker links between otherwise dense areas. In social-network terms, these are the ties that can carry information from one friend circle to another. This matches Granovetter's idea of the **strength of weak ties**: a bridge may not look important by degree, but it can be important because it connects social circles that would otherwise be separated.


## 6. Louvain Method Implementation

Louvain optimizes modularity, so it is usually fast on graphs of this size and tends to find communities with dense internal connections.


In [11]:
louvain_start = time.perf_counter()
louvain_partition = community_louvain.best_partition(G, random_state=RANDOM_SEED)
louvain_runtime = time.perf_counter() - louvain_start

louvain_communities_by_id = {}
for node, community_id in louvain_partition.items():
    louvain_communities_by_id.setdefault(community_id, set()).add(node)

louvain_communities = list(louvain_communities_by_id.values())
louvain_sizes = sorted([len(nodes) for nodes in louvain_communities], reverse=True)
louvain_modularity = modularity(G, louvain_communities)

louvain_summary = pd.DataFrame(
    [
        ["Number of communities", len(louvain_communities)],
        ["Modularity score", louvain_modularity],
        ["Runtime seconds", louvain_runtime],
        ["Community sizes descending", louvain_sizes],
    ],
    columns=["Metric", "Value"],
)
display(louvain_summary)

louvain_size_table = pd.DataFrame(
    sorted(
        [{"community": cid, "size": len(nodes)} for cid, nodes in louvain_communities_by_id.items()],
        key=lambda row: row["size"],
        reverse=True,
    )
)
display(louvain_size_table)


,Metric,Value
0,Number of communities,11
1,Modularity score,0.400862
2,Runtime seconds,0.080397
3,Community sizes descending,"[127, 117, 116, 110, 101, 95, 79, 17, 3, 2, 2]"


,community,size
0,2,127
1,8,117
2,0,116
3,4,110
4,1,101
5,6,95
6,3,79
7,7,17
8,5,3
9,9,2


In [12]:
louvain_node_colors = [louvain_partition[node] for node in G_lcc.nodes()]

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(G_lcc, pos_lcc, ax=ax, width=0.25, alpha=0.10, edge_color="#555555")
nx.draw_networkx_nodes(
    G_lcc,
    pos_lcc,
    ax=ax,
    node_color=louvain_node_colors,
    node_size=22,
    cmap="tab20",
    linewidths=0,
    alpha=0.95,
)
ax.set_title(f"Louvain communities on {DATASET_NAME} (modularity={louvain_modularity:.3f})")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "08_louvain_communities.png", dpi=180)
plt.close(fig)


![Louvain communities](figures/08_louvain_communities.png)


### Louvain Result in This Dataset

The Louvain method found **11 communities** with a modularity score of **0.4009** in about **0.08 seconds**. The community sizes are **[127, 117, 116, 110, 101, 95, 79, 17, 3, 2, 2]**. This size pattern is believable for a campus Facebook network because it does not simply split the graph into one huge component and many singletons. Instead, it finds several medium-sized groups between about 79 and 127 users, plus a few small peripheral communities.

The modularity value is also meaningful. A score around **0.40** indicates that the detected groups have noticeably denser internal connections than would be expected by chance. In Figure 8, the colored Louvain groups occupy recognizable regions in the spring layout, which suggests that the algorithm is capturing real local structure rather than assigning colors randomly. In the Caltech context, these communities may correspond to residence clusters, year groups, friend circles, clubs, or overlapping academic/social environments. The dataset does not include node attributes, so those labels cannot be proven directly, but the community sizes and modularity make this interpretation reasonable.


## 7. Girvan-Newman Method Implementation

Girvan-Newman repeatedly removes edges with high edge betweenness. The full exact algorithm can be slow on dense graphs, so this notebook uses NetworkX's Girvan-Newman implementation with a fixed-seed approximate edge-betweenness scorer. The method is applied to the largest connected component so that `k = 2, 3, 4` partitions are meaningful.


In [13]:
def approximate_most_valuable_edge(graph: nx.Graph):
    sample_size = min(80, max(10, int(graph.number_of_nodes() ** 0.5 * 4)))
    edge_scores = nx.edge_betweenness_centrality(graph, k=sample_size, seed=RANDOM_SEED)
    return max(edge_scores, key=edge_scores.get)


target_k_values = {2, 3, 4}
girvan_newman_partitions = {}
girvan_newman_rows = []

gn_start = time.perf_counter()
for communities_tuple in girvan_newman(G_lcc, most_valuable_edge=approximate_most_valuable_edge):
    communities = [set(community) for community in communities_tuple]
    community_count = len(communities)

    if community_count in target_k_values:
        partition_runtime = time.perf_counter() - gn_start
        community_sizes = sorted([len(community) for community in communities], reverse=True)
        partition_modularity = modularity(G_lcc, communities)
        girvan_newman_partitions[community_count] = communities
        girvan_newman_rows.append(
            {
                "k": community_count,
                "modularity": partition_modularity,
                "runtime_seconds": partition_runtime,
                "community_sizes": community_sizes,
            }
        )

    if community_count >= max(target_k_values):
        break

girvan_newman_summary = pd.DataFrame(girvan_newman_rows)
display(girvan_newman_summary)


,k,modularity,runtime_seconds,community_sizes
0,2,-1.803389e-09,0.286162,"[761, 1]"
1,3,-5.410166e-09,0.531258,"[760, 1, 1]"
2,4,-1.082033e-08,0.769575,"[759, 1, 1, 1]"


In [14]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

for ax, k in zip(axes, [2, 3, 4]):
    communities = girvan_newman_partitions[k]
    node_to_community = {
        node: index
        for index, community in enumerate(communities)
        for node in community
    }
    node_colors = [node_to_community[node] for node in G_lcc.nodes()]
    community_sizes = sorted([len(community) for community in communities], reverse=True)
    modularity_value = float(girvan_newman_summary.loc[girvan_newman_summary["k"] == k, "modularity"].iloc[0])

    nx.draw_networkx_edges(G_lcc, pos_lcc, ax=ax, width=0.20, alpha=0.08, edge_color="#555555")
    nx.draw_networkx_nodes(
        G_lcc,
        pos_lcc,
        ax=ax,
        node_color=node_colors,
        node_size=20,
        cmap="Set2",
        linewidths=0,
        alpha=0.95,
    )
    ax.set_title(f"Girvan-Newman k={k}\nmodularity={modularity_value:.4f}; sizes={community_sizes[:4]}")
    ax.axis("off")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "09_girvan_newman_communities.png", dpi=180)
plt.close(fig)


![Girvan-Newman communities](figures/09_girvan_newman_communities.png)


### Girvan-Newman Result in This Dataset

The Girvan-Newman output is very different from the Louvain output. For **k = 2**, it produced community sizes **[761, 1]** with modularity approximately **0**. For **k = 3**, the size pattern became **[760, 1, 1]**, and for **k = 4** it became **[759, 1, 1, 1]**. This means the algorithm did not discover two, three, or four balanced social communities. Instead, it repeatedly peeled off single peripheral nodes from the main component.

This behavior makes sense once we consider the graph structure. Girvan-Newman is a divisive algorithm: it looks for high edge-betweenness edges and removes them. That works well when a network has clear bottleneck edges between large groups. However, `socfb-Caltech36` is dense, highly clustered, and full of alternative paths. The true social groups are not separated by a few obvious bottlenecks. As a result, the earliest Girvan-Newman splits are dominated by fragile peripheral attachments rather than meaningful large-scale communities.

So, in this dataset, Girvan-Newman is still useful as a diagnostic tool because it reveals weakly attached nodes and bridge-like edges. But for community detection, the first few partitions are not socially convincing. The near-zero modularity values confirm this: these partitions do not improve the separation of dense friendship groups.


## 8. Comparative Analysis


In [15]:
comparison_rows = [
    {
        "method": "Louvain",
        "graph_used": "Full graph",
        "communities": len(louvain_communities),
        "modularity": louvain_modularity,
        "runtime_seconds": louvain_runtime,
        "largest_community_size": max(louvain_sizes),
        "smallest_community_size": min(louvain_sizes),
        "size_pattern": str(louvain_sizes),
    }
]

for _, row in girvan_newman_summary.iterrows():
    sizes = row["community_sizes"]
    comparison_rows.append(
        {
            "method": f"Girvan-Newman k={int(row['k'])}",
            "graph_used": "Largest connected component",
            "communities": int(row["k"]),
            "modularity": float(row["modularity"]),
            "runtime_seconds": float(row["runtime_seconds"]),
            "largest_community_size": max(sizes),
            "smallest_community_size": min(sizes),
            "size_pattern": str(sizes),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(comparison_df["method"], comparison_df["modularity"], color=["#2a9d8f", "#f4a261", "#f4a261", "#f4a261"])
axes[0].set_title("Modularity comparison")
axes[0].set_ylabel("Modularity")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(comparison_df["method"], comparison_df["runtime_seconds"], color=["#2a9d8f", "#e76f51", "#e76f51", "#e76f51"])
axes[1].set_title("Runtime comparison")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=25)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "10_method_comparison.png", dpi=180)
plt.close(fig)


,method,graph_used,communities,modularity,runtime_seconds,largest_community_size,smallest_community_size,size_pattern
0,Louvain,Full graph,11,4.008625e-01,0.080397,127,2,"[127, 117, 116, 110, 101, 95, 79, 17, 3, 2, 2]"
1,Girvan-Newman k=2,Largest connected component,2,-1.803389e-09,0.286162,761,1,"[761, 1]"
2,Girvan-Newman k=3,Largest connected component,3,-5.410166e-09,0.531258,760,1,"[760, 1, 1]"
3,Girvan-Newman k=4,Largest connected component,4,-1.082033e-08,0.769575,759,1,"[759, 1, 1, 1]"


![Method comparison](figures/10_method_comparison.png)


### Method Comparison and Final Interpretation

The two algorithms give almost opposite results on the same graph. **Louvain** found **11 communities** with modularity **0.4009**, while the tested **Girvan-Newman** partitions for **k = 2, 3, 4** all had modularity values extremely close to **0**. This is the strongest quantitative evidence that Louvain fits this dataset better.

The reason is not only runtime, although runtime also favors Louvain. Louvain finished in about **0.08 seconds**, while the approximate Girvan-Newman run took about **0.29 seconds** for k = 2, **0.53 seconds** for k = 3, and **0.77 seconds** for k = 4. More importantly, Louvain produced communities with interpretable sizes: **127, 117, 116, 110, 101, 95, 79**, plus a few small groups. Girvan-Newman produced **[761, 1]**, then **[760, 1, 1]**, then **[759, 1, 1, 1]**. Those Girvan-Newman splits are mathematically valid, but they are not useful if the goal is to describe real student communities.

The difference comes from the algorithm mechanisms. Louvain is a modularity-based method, so it searches for groups where internal friendship density is stronger than expected. That matches the structure of this graph: high clustering, many triangles, and several dense friendship regions. Girvan-Newman is edge-betweenness based, so it needs clear bridge edges between major groups. In this Caltech network, there are bridges and local bridges, but the main social core has many redundant paths. Therefore, the algorithm first removes peripheral single-node attachments instead of cutting the graph into meaningful large communities.

For the social interpretation, Louvain is therefore the better result to discuss. Its communities are plausibly related to residence houses, class-year circles, clubs, shared courses, or friendship groups. Since the dataset only gives edges and not node attributes, I cannot prove the exact real-world label of each community. Still, the combination of **modularity 0.4009**, medium-sized communities, and clear visual grouping in Figure 8 makes Louvain much more convincing for this network than the early Girvan-Newman splits.

Overall, `socfb-Caltech36` behaves like a compact and highly connected campus friendship network. It has a strong central hub around node **709**, short path lengths, high clustering, and many triangles. The best community detection method for this structure is Louvain because it respects dense local friendship groups instead of simply cutting off weak peripheral nodes.
